#### Ingest pipeline perform common transformations such as remove fields, extract values from text on data before indexing. A pipeline consists of a series of configurable tasks called processors. Each processor runs sequentially, making specific changes to incoming documents. After the processors have run, Elasticsearch adds the transformed documents to data stream or index

In [1]:
from pprint import pprint
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
client_info = es.info()
pprint('Connected to Elasticsearch successfully!')
pprint(client_info.body)

'Connected to Elasticsearch successfully!'
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'WGXTdf8bTw6Y1ejhBBncsA',
 'name': 'c813a54bbd9a',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.15.0'}}


#### Create the pipeline

In [4]:
from pprint import pprint

response = es.ingest.put_pipeline(
    id='lowercase_text',
    description='Transforms the text to lowercase',
    processors=[
        {
            "lowercase": {
                "field": "text"
            }
        }
    ]
)
pprint(response.body)

{'acknowledged': True}


#### Get the pipeline

In [5]:
response = es.ingest.get_pipeline(id='lowercase_text')
pprint(response.body)

{'lowercase_text': {'description': 'Transforms the text to lowercase',
                    'processors': [{'lowercase': {'field': 'text'}}]}}


#### Delete the pipeline

In [6]:
response = es.ingest.delete_pipeline(id='lowercase_text')
pprint(response.body)

{'acknowledged': True}


#### Simulate to test before applying the pipeline to real index and data

In [7]:
response = es.ingest.put_pipeline(
    id='lowercase_text',
    description='Transforms the text to lowercase',
    processors=[
        {
            "lowercase": {
                "field": "text"
            }
        }
    ]
)
pprint(response.body)

{'acknowledged': True}


putting test data inside the docs list, as this is just the simulation, nothing will be indexed and we can have the preview how the data will look after applying the pipeline

In [11]:
response = es.ingest.simulate(
    id='lowercase_pipeline',
    docs=[
        {
            "_index": "my_index",
            "_id": "1",
            "_source": {
                "text": "TWINKLING STARS"
            }
        }
    ]
)
pprint(response.body)

{'docs': [{'doc': {'_id': '1',
                   '_index': 'my_index',
                   '_ingest': {'timestamp': '2026-05-23T17:25:34.503108947Z'},
                   '_source': {'text': 'twinkling stars'},
                   '_version': '-3'}}]}


now using the pipline to transform the text to lowercase

In [14]:
import json

dummy_data = json.load(open("data/dummy_data.json"))
for i, document in enumerate(dummy_data):
    document['text'] = document['text'].lower()
    dummy_data[i] = document

dummy_data

[{'title': 'Title 1', 'text': 'description 1', 'created_on': '2026-05-01'},
 {'title': 'Title 2', 'text': 'description 2', 'created_on': '2026-05-02'},
 {'title': 'Title 3', 'text': 'description 3', 'created_on': '2026-05-03'},
 {'title': 'Title 4', 'text': 'description 4', 'created_on': '2026-05-04'},
 {'title': 'Title 5', 'text': 'description 5', 'created_on': '2026-05-05'}]

In [15]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

### Now we pass the `lowercase_text`pipeline to the bulk method. It will perform the transformations before indexing the documents

In [17]:
operations = []
for document in dummy_data:
    operations.append({'index': {'_index': 'my_index'}})
    operations.append(document)

response = es.bulk(operations=operations, pipeline='lowercase_text')
pprint(response.body)

{'errors': False,
 'ingest_took': 1,
 'items': [{'index': {'_id': 'sP3hVZ4BSLzgL3Jkt7SZ',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 5,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
                      'result': 'created',
                      'status': 201}},
           {'index': {'_id': 'sf3hVZ4BSLzgL3Jkt7SZ',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 6,
                      '_shards': {'failed': 0, 'successful': 1, 'total': 2},
                      '_version': 1,
                      'result': 'created',
                      'status': 201}},
           {'index': {'_id': 'sv3hVZ4BSLzgL3Jkt7SZ',
                      '_index': 'my_index',
                      '_primary_term': 1,
                      '_seq_no': 7,
                      '_shards': {'failed': 0,

##### After indexing the documents, the text field for all documents has been lowercased. This shows the pipline has run without any errors.

In [19]:
response = es.search(index='my_index')
hits = response.body['hits']['hits']

for hit in hits:
    print(hit['_source'])

{'title': 'Title 1', 'created_on': '2026-05-01', 'text': 'description 1'}
{'title': 'Title 2', 'created_on': '2026-05-02', 'text': 'description 2'}
{'title': 'Title 3', 'created_on': '2026-05-03', 'text': 'description 3'}
{'title': 'Title 4', 'created_on': '2026-05-04', 'text': 'description 4'}
{'title': 'Title 5', 'created_on': '2026-05-05', 'text': 'description 5'}
{'title': 'Title 1', 'created_on': '2026-05-01', 'text': 'description 1'}
{'title': 'Title 2', 'created_on': '2026-05-02', 'text': 'description 2'}
{'title': 'Title 3', 'created_on': '2026-05-03', 'text': 'description 3'}
{'title': 'Title 4', 'created_on': '2026-05-04', 'text': 'description 4'}
{'title': 'Title 5', 'created_on': '2026-05-05', 'text': 'description 5'}
